# Setup

In [ ]:
import os
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm

In [ ]:
MLST = '../../data/processed/mlst_report.tsv'
METADATA = '../../data/metadata/mash_scrubbed_species_metadata.csv'

In [ ]:
mlst = pd.read_csv(MLST, sep='\t', header=None, dtype='object')

# Add column names
mlst.columns = [
    'genome_id',
    'schema',
    'mlst',
    'dnaA',
    'fusA',
    'gyrB',
    'leuS',
    'pyrG',
    'rplB',
    'rpoB'
]

mlst

# Enrich metadata

For now, its just MLST. Add in other things as needed

In [ ]:
mash_scrubbed_metadata = pd.read_csv(METADATA, index_col=0, dtype='object')

display(
    mash_scrubbed_metadata.shape,
    mash_scrubbed_metadata.head()
)

In [ ]:
mash_scrubbed_metadata['mlst'] = None

for idx in tqdm(mash_scrubbed_metadata.index):
    genome_id = mash_scrubbed_metadata.loc[idx, 'genome_id']
    mlst_value = mlst.set_index('genome_id').loc[f'{genome_id}.fna', 'mlst']

    # if non-exact mlst allele match, set to -1
    if mlst_value == '-':
        mlst_value = -1
    
    mash_scrubbed_metadata.loc[idx, 'mlst'] = mlst_value

mash_scrubbed_metadata.head()

## Add GTDB-tk information

In [ ]:
BAKTA = '../../data/processed/bakta/'

bakta_fna_paths = [
    os.path.join(BAKTA, bakta_folder, bakta_folder+'.fna')
    for bakta_folder in os.listdir(BAKTA)
]

In [ ]:
bakta_fna_paths = [x.split('/', maxsplit=2)[-1] for x in bakta_fna_paths]

In [ ]:
with open('../../data/processed/gtdb_output/genome_fna_paths.txt', 'w') as f:
    for path in bakta_fna_paths:
        genome = path.split('/')[-1][:-4]
        to_write = path + '\t' + genome + '\n'
        f.write(to_write)

Run GTDB-tk on the dataset using the following command from the home directory of this repository

`gtdbtk classify_wf --batchfile data/processed/gtdb_output/genome_fna_paths.txt --out_dir data/processed/gtdb_output/`

In [ ]:
gtdb_results = pd.read_csv('../../data/processed/gtdb_output/gtdbtk.bac120.summary.tsv', sep = '\t', dtype ='object')
mash_scrubbed_metadata['closest_genome_reference'] = gtdb_results.set_index('user_genome').loc[mash_scrubbed_metadata.genome_id.values, 'closest_genome_reference'].values
mash_scrubbed_metadata['closest_genome_taxonomy'] = gtdb_results.set_index('user_genome').loc[mash_scrubbed_metadata.genome_id.values, 'closest_genome_taxonomy'].values

# Save enriched metadata file

In [ ]:
mash_scrubbed_metadata.to_csv('../../data/metadata/enriched_metadata.csv')